# 04 — Feature Engineering and Time-Series Preparation (Stages 4–5)

1. Aggregate SALE rows to **daily demand per product**.
2. Choose the 500 products with the most sale days (enough history to learn from).
3. Build a product × calendar-day **panel** from each product's first to last sale (no rows outside that window).
4. Create lag, rolling, calendar, price and history features using **only past data** (`shift(1)` before rolling).
5. Split by **date**: first 70% train, next 15% validation, last 15% test.

Target `NEXT_DAY_DEMAND`: for a row dated D, the units sold on D, predicted from information up to the end of D-1.

> **Colab:** run the setup cell; it asks for the project ZIP the first time. Put `Online Retail.xlsx` in `data/raw/` (or run notebook 01 first, which uploads it).

In [1]:
import sys, os, zipfile, io
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd()
for c in (ROOT, ROOT.parent, ROOT / "AI-Sales-Forecasting-Inventory"):
    if (c / "ml").is_dir():
        ROOT = c
        break
if IN_COLAB and not (ROOT / "ml").is_dir():
    from google.colab import files
    print("Upload the project ZIP once (AI-Sales-Forecasting-Inventory.zip):")
    for name, data in files.upload().items():
        zipfile.ZipFile(io.BytesIO(data)).extractall("/content")
    ROOT = Path("/content/AI-Sales-Forecasting-Inventory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["SFIO_PROJECT_ROOT"] = str(ROOT)

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Image, display, Markdown
from ml import config
config.ensure_directories()
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
print("Project root:", ROOT.name, "| Colab:", IN_COLAB)

Project root: AI-Sales-Forecasting-Inventory | Colab: False


In [2]:
from ml.preprocessing import load_clean_sales
from ml import feature_engineering as fe
sales = load_clean_sales()
out = fe.build_dataset(sales)
print("Products modelled:", len(out["products"]))
print("Panel rows:", f"{len(out['panel']):,}", "| model rows:", f"{len(out['model_data']):,}")
print("Weekdays the shop never traded (0=Mon):", fe.closed_weekdays(out["calendar"]))

Products modelled: 500
Panel rows: 182,125 | model rows: 138,700
Weekdays the shop never traded (0=Mon): [5]


## Leakage check on one product
Look at the columns: `lag_1` on a row equals the previous row's demand, and `rolling_mean_7` uses the 7 days **before** the row, never the row itself.

In [3]:
code = out["products"][0]
ex = out["model_data"][out["model_data"]["StockCode"] == code]
ex[["Date", "NEXT_DAY_DEMAND", "lag_1", "lag_7", "rolling_mean_7", "rolling_max_7", "last_known_price", "days_since_prev_sale"]].head(10)

,Date,NEXT_DAY_DEMAND,lag_1,lag_7,rolling_mean_7,rolling_max_7,last_known_price,days_since_prev_sale
181037,2011-01-04,67.0,0.0,0.0,0.000000,0.0,2.677660,12.0
181038,2011-01-05,304.0,67.0,0.0,9.571429,67.0,2.803134,1.0
181039,2011-01-06,171.0,304.0,0.0,53.000000,304.0,2.613158,1.0
181040,2011-01-07,334.0,171.0,0.0,77.428571,304.0,2.650585,1.0
181042,2011-01-09,97.0,0.0,0.0,125.142857,334.0,2.612275,2.0
181043,2011-01-10,12.0,97.0,0.0,139.000000,334.0,2.818041,1.0
181044,2011-01-11,1184.0,12.0,67.0,140.714286,334.0,2.950000,1.0
181045,2011-01-12,82.0,1184.0,304.0,300.285714,1184.0,3.154139,1.0
181046,2011-01-13,8.0,82.0,171.0,268.571429,1184.0,2.902195,1.0
181047,2011-01-14,89.0,8.0,334.0,245.285714,1184.0,3.660000,1.0


In [4]:
p = out["panel"][out["panel"]["StockCode"] == code].reset_index(drop=True)
row = 40
manual = p["demand"].iloc[row - 7: row].mean()
feat = fe.make_features(p).loc[row, "rolling_mean_7"]
print(f"rolling_mean_7 on {p.loc[row, 'Date'].date()}: feature={feat:.3f}, mean of the 7 previous days={manual:.3f}")
assert abs(manual - feat) < 1e-9

rolling_mean_7 on 2011-01-10: feature=139.000, mean of the 7 previous days=139.000


## Feature dictionary

In [5]:
pd.DataFrame(fe.FEATURE_DICTIONARY).T

,family,description
lag_1,lag,Units sold on the previous calendar day (D-1)....
lag_7,lag,Units sold 7 days earlier (same weekday last w...
lag_14,lag,Units sold 14 days earlier.
lag_28,lag,Units sold 28 days earlier.
last_open_day_demand,lag,Units sold on the most recent day the shop was...
prev_day_open,calendar,"1 if the shop was open on D-1, else 0."
rolling_mean_7,rolling,Mean daily units over D-7..D-1 (shift(1) then ...
rolling_mean_14,rolling,Mean daily units over D-14..D-1.
rolling_mean_28,rolling,Mean daily units over D-28..D-1.
rolling_std_7,rolling,Standard deviation of daily units over D-7..D-...


## Chronological split (by date)

In [6]:
info = out["split_info"]
display(pd.DataFrame({k: info[k] for k in ("train", "validation", "test")}).T)
print(info["rows"])
d = out["model_data"]
assert d.loc[d.split == "train", "Date"].max() < d.loc[d.split == "validation", "Date"].min() < d.loc[d.split == "test", "Date"].min()

,start,end
train,2011-01-04T00:00:00,2011-08-31T00:00:00
validation,2011-09-01T00:00:00,2011-10-20T00:00:00
test,2011-10-21T00:00:00,2011-12-09T00:00:00


{'train': 96328, 'validation': 21472, 'test': 20900}


In [7]:
res = fe.run(sales)   # saves panel, features, feature_columns.json and docs/FEATURE_DICTIONARY.md
print("Saved data/processed/features.parquet and artifacts/feature_columns.json")

Saved data/processed/features.parquet and artifacts/feature_columns.json
